# Customer Churn - Telecom
---

### CRISP-DM Methodology
This project follows the CRISP-DM (*Cross-Industry Standard Process for Data Mining*) framework applied to **Customer Retention & Churn Prediction**:
| **Stage** | **Objective** | **Methodological Execution** |
| :--- | :--- | :--- |
| **1. Business Understanding** | Mitigate revenue loss by identifying at-risk customers. | • **Target Definition**: Binary Classification (Churn: Yes/No).<br>• **KPIs**: Maximize **Lift** in retention campaigns & Revenue Saved vs. Cost. |
| **2. Data Understanding** | Detect patterns of friction and dissatisfaction. | • **EDA**: Distribution analysis (Detect Imbalance).<br>• **Hypothesis Testing**: Correlation Matrix & Independence Tests (Chi-Square). |
| **3. Data Preparation** | Construct a robust dataset for parametric modeling. | • **Scaling**: Standardization (Z-score) for coefficient comparability.<br>• **Encoding**: One-Hot Encoding for nominal variables.<br>• **Splitting**: Stratified Train/Test Split to preserve class ratio. |
| **4. Modeling** | Estimate Churn Probability | • **Algorithms**: Logistic Regression, LinearSVC, KNN, Random Florest, XGBoost, LightGBM.<br>• **Inference**: Analyze **Odds Ratios** to determine feature elasticity. |
| **5. Evaluation** | Assess model reliability and financial impact. | • **Discrimination**: AUC-ROC & F1-Score (Threshold Tuning).<br>• **Calibration**: Probability Calibration Curve (Reliability Diagram). |
| **6. Deployment** | Integrate insights into the CRM lifecycle. | • **Deliverable**: "High-Risk" Customer List for Marketing Squad.<br>• **Artifact**: Serialize model (`joblib`) for batch inference. |

---

### Installs:

In [0]:
%%capture
%pip install -r '../requirements.txt'
# Command to restart the kernel and update the installed libraries
%restart_python

### Imports:

In [0]:
# ======================================================== #
# Standard library
# ======================================================== #
import warnings

import joblib


# ======================================================== #
# Third-party libraries - data analysis and visualization
# ======================================================== #
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# ======================================================== #
# Third-party libraries - modeling and preprocessing
# ======================================================== #
import optuna
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier


warnings.filterwarnings("ignore")


In [0]:
# SRC/ Functions Utils:
import sys
sys.path.append('../src')
from visualization import GraphicsData
from utils import EDATest
from modeling_utils import DtypeOptimizer, FeatureEngineer, classification_kfold_cv, RecursiveFeatureEliminator

### Dev objects

In [0]:
from matplotlib.gridspec import GridSpec

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    brier_score_loss,
    classification_report
)

def evaluation_scores(
    y_true,
    y_score,
    X = None,
    pos_label = 1,
    threshold = 0.5,
    figsize = (6, 12),
    title_prefix = ""
):
    """
    """

    try:

        y_true = np.asarray(y_true).ravel()
        y_score = np.asarray(y_score).ravel()

        # Check Entry
        if len(y_true) != len(y_score):
            raise ValueError('y_true and y_score need to be the same size.')

        unique_labels = np.unique(y_true)
        if len(unique_labels) != 2:
            raise ValueError(f'The function only supports binary classification. Labels found: {unique_labels}')

        y_bin = (y_true == pos_label).astype(int)
        
        # Scores 
        fpr, tpr, _ = roc_curve(y_bin, y_score)
        auroc_value = roc_auc_score(y_bin, y_score)

        precision_vals, recall_vals, _ = precision_recall_curve(y_bin, y_score)
        ap_value = average_precision_score(y_bin, y_score)

        prevalence = y_bin.mean()

        # Binary Prediction
        y_pred = (y_score >= threshold).astype(int)

        # Confusion matrix
        cm = confusion_matrix(y_bin, y_pred)

        # Metrics threshold-based
        accuracy = accuracy_score(y_bin, y_pred)
        precision = precision_score(y_bin, y_pred, zero_division = 0)
        recall = recall_score(y_bin, y_pred, zero_division = 0)
        f1 = f1_score(y_bin, y_pred, zero_division = 0)

        # Metrics scores-based
        roc_auc = auroc_value
        pr_auc = ap_value
        brier = brier_score_loss(y_bin, y_score)
        ks = np.max(tpr - fpr)
        gini = 2 * roc_auc - 1

        # Classification report
        report_dict = classification_report(
            y_bin,
            y_pred,
            output_dict = True,
            zero_division = 0
        )
        report_df = pd.DataFrame(report_dict).T.reset_index()
        report_df = report_df.rename(columns = {'index': 'class'})
        report_df = report_df.round(4)

        desired_order = ['0', '1', 'accuracy', 'macro avg', 'weighted avg']
        report_df['order'] = report_df['class'].apply(
            lambda x: desired_order.index(x) if x in desired_order else 999
        )

        report_df = (
            report_df
            .sort_values('order')
            .drop(columns = 'order')
            .reset_index(drop = True)
        )

        # Metrics Df
        metrics_df = pd.DataFrame({
            'Metric': [
                'Accuracy',
                'Precison',
                'Recall',
                'F1-Score',
                'ROC-AUC',
                'KS',
                'Gini',
                'PR-AUC',
                'Brier'
            ],
            'Value':[
                accuracy,
                precision,
                recall,
                f1,
                roc_auc,
                ks,
                gini,
                pr_auc,
                brier
            ]
        })

        metrics_df['Value'] = metrics_df['Value'].round(4)

        # Graphics
        
        fig = plt.figure(figsize = figsize)
        gs = GridSpec(3, 1, figure = fig, height_ratios=[1, 1, 1.15])

        ax_roc = fig.add_subplot(gs[0, 0])
        ax_pr = fig.add_subplot(gs[1, 0])
        ax_cm = fig.add_subplot(gs[2, 0])

        # ROC
        ax_roc.plot(
            fpr, tpr,
            color = '#1f77b4',
            lw=2,
            label = f'ROC (AUROC = {auroc_value:.3f})'
        )
        ax_roc.plot(
            [0, 1], [0, 1],
            color = 'gray',
            linestyle = '--',
            label = 'Random'
        )
        ax_roc.set_xlabel('False Positive Rate', fontsize = 12)
        ax_roc.set_ylabel('True Positive Rate', fontsize = 12)
        ax_roc.set_title(f'{title_prefix}ROC Curve', fontsize=14, fontweight = 'bold')
        ax_roc.legend(loc = 'lower right', fontsize = 10)
        ax_roc.grid(True, linestyle = '--', linewidth = 0.5)

        # PR
        ax_pr.plot(
            recall_vals, precision_vals,
            color = 'darkorange',
            lw=2,
            label = f'PR Curve (AP = {ap_value:.3f})'
        )
        ax_pr.axhline(
            prevalence,
            color = 'gray',
            linestyle = '--',
            label = f'Baseline = {prevalence:.3f}'
        )
        ax_pr.set_xlabel('Recall', fontsize = 12)
        ax_pr.set_ylabel('Precision', fontsize = 12)
        ax_pr.set_title(
            f'{title_prefix}Precision-Recall Curve',
            fontsize = 14,
            fontweight = 'bold'
        )
        ax_pr.legend(loc = 'upper right', fontsize = 10)
        ax_pr.grid(True, linestyle = '--', linewidth = 0.5)

        # Confusion Matrix
        disp = ConfusionMatrixDisplay(
            confusion_matrix = cm,
            display_labels = [0, 1]
        )
        disp.plot(ax = ax_cm, cmap = 'viridis', colorbar = False)
        ax_cm.grid(False)
        ax_cm.set_title(
            f'{title_prefix}Confusion Matrix (threshold = {threshold:.2f})',
            fontsize = 14,
            fontweight = 'bold'
        )

        plt.tight_layout()
        plt.show()

        return report_df, metrics_df

    except Exception as e:
        raise RuntimeError(f'Failure to generate ROC curves, PR, confusion matrix and classification report: {e}') from e

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def plot_kde_predictions(
    y_true,
    y_score,
    palette=('#12e193', '#feb308'),
    title='Prediction Probabilities',
    n_bins=10,
    random_state=42,
    figsize=(12, 8)
):
    """
    Plota:
    1. Distribuição KDE das probabilidades preditas por classe.
    2. Ordenação dos scores por decis com taxa do evento.

    Parâmetros
    ----------
    y_true : array-like
        Labels reais binários.
    y_score : array-like
        Probabilidades preditas da classe positiva.
    palette : tuple/list, opcional
        Cores usadas no KDE por classe.
    title : str, opcional
        Título do gráfico KDE.
    n_bins : int, opcional
        Número desejado de bins/decis.
    random_state : int, opcional
        Seed para o ruído usado no desempate no qcut.
    figsize : tuple, opcional
        Tamanho da figura.

    Retorna
    -------
    fig, axes, decile_df
    """
    try:
        y_true = np.asarray(y_true).ravel()
        y_score = np.asarray(y_score).ravel()

        if len(y_true) != len(y_score):
            raise ValueError("y_true e y_score precisam ter o mesmo tamanho.")

        if len(np.unique(y_true)) != 2:
            raise ValueError("A função suporta apenas classificação binária.")

        plot_df = pd.DataFrame({
            'y_true': y_true,
            'y_score': y_score
        }).dropna().copy()

        if plot_df.empty:
            raise ValueError("Não há dados válidos após remover valores nulos.")

        plot_df['y_true'] = pd.to_numeric(plot_df['y_true'], errors='coerce')
        plot_df['y_score'] = pd.to_numeric(plot_df['y_score'], errors='coerce')
        plot_df = plot_df.dropna().copy()

        if plot_df.empty:
            raise ValueError("Os dados ficaram vazios após conversão numérica.")

        rng = np.random.default_rng(random_state)
        noise = rng.uniform(0, 1e-4, size=len(plot_df))
        plot_df['score_adj'] = plot_df['y_score'].to_numpy() + noise

        plot_df = plot_df.sort_values(by='score_adj', ascending=True).reset_index(drop=True)

        plot_df['decile'] = pd.qcut(
            plot_df['score_adj'],
            q=n_bins,
            labels=False,
            duplicates='drop'
        )

        decile_df = (
            plot_df
            .groupby('decile', observed=True)['y_true']
            .mean()
            .reset_index()
            .rename(columns={'y_true': 'event_rate'})
        )

        decile_df['decile'] = decile_df['decile'].astype(int) + 1

        plt.rc('font', size=10)
        fig, axes = plt.subplots(
            2, 1,
            figsize=figsize,
            gridspec_kw={'height_ratios': [1.2, 1]}
        )

        ax_kde, ax_bar = axes

        sns.kdeplot(
            data=plot_df,
            x='y_score',
            hue='y_true',
            fill=True,
            alpha=0.4,
            bw_adjust=1,
            palette=palette,
            linewidth=1,
            cut=0,
            ax=ax_kde
        )

        ax_kde.set_title(title, fontsize=14, fontweight='bold')
        ax_kde.set_xlabel('Predicted probability')
        ax_kde.set_ylabel(None)
        ax_kde.set_yticklabels([])
        ax_kde.grid(axis='y', linestyle='--', linewidth=0.3)
        ax_kde.grid(axis='x', linestyle='--', linewidth=0.3)
        sns.despine(ax=ax_kde, top=True, right=True, left=True, bottom=False)

        bars = ax_bar.bar(
            decile_df['decile'],
            decile_df['event_rate'],
            color='#023047',
            edgecolor='white',
            linewidth=0.8
        )

        ax_bar.set_title(
            'Probability scores ordering - Event rate per decile',
            loc='left',
            fontweight='bold',
            fontsize=13
        )
        ax_bar.set_xlabel('Decile', labelpad=12)
        ax_bar.set_ylabel(None)
        ax_bar.set_xticks(decile_df['decile'])
        ax_bar.tick_params(axis='both', which='both', length=0)
        ax_bar.yaxis.set_visible(False)

        ax_bar.spines['top'].set_visible(False)
        ax_bar.spines['right'].set_visible(False)
        ax_bar.spines['left'].set_visible(False)
        ax_bar.grid(False)

        offset = max(decile_df['event_rate'].max() * 0.03, 0.005)

        for bar, event_rate in zip(bars, decile_df['event_rate']):
            height = bar.get_height()
            ax_bar.text(
                bar.get_x() + bar.get_width() / 2,
                height + offset,
                f'{event_rate * 100:.1f}%',
                ha='center',
                va='bottom',
                color='black',
                fontsize=10,
                fontweight='bold'
            )

        plt.tight_layout()
        plt.show()

        return fig, axes, decile_df

    except Exception as e:
        raise RuntimeError(
            f"Failed to generate prediction probability graphs: {str(e)}."
        ) from e


### Load the data

In [0]:
df = pd.read_csv('../data/BankChurners.csv')

### Verify successful load with some randomly selected records


In [0]:
df.sample(9)

In [0]:
df.info()


### 3 - Data Preparation

In [0]:
GraphicsData(data = df).plot_target_analysis(target_col='Attrition_Flag', colors=['#1abc9c', '#ff6b6b'])

##### Methodological Note:
---

- Prior to conducting the bivariate analysis, the dataset will be partitioned into **Train** and **Test** sets.  
The primary objective is to prevent **Data Leakage**, ensuring that all statistical insights, outlier treatments, transformation decisions, and feature engineering strategies are derived exclusively from the training data.

> This approach preserves the integrity of the validation process and guarantees that performance metrics reflect true generalization capacity rather than information contamination.

- The `stratify` parameter will be applied within the `train_test_split` procedure.  
Given that churn prediction is a **binary classification problem**, maintaining the original **class proportion** across both subsets is statistically mandatory.

> Stratified sampling preserves the prior probability distribution of the target variable, avoiding distortions in class balance that could bias model training, threshold calibration, and performance evaluation metrics (e.g., Recall, Precision, ROC-AUC).

---


##### Adjusting the target variable
---

I will adjust the target column, `Attrition_Flag`. Since this is a categorical variable with binary classification:

---

- **Existing Customer (Non-churner)**: will receive the **value 0**.

  These are customers who still use the credit card services provided by the bank.

---

- **Attrited Customer (Churner)**: will receive the **value 1**.

---

> This approach will facilitate the calculation of correlation statistics between variables, since the target variable will already be properly encoded.

---



In [0]:
map_churn = {
    'Existing Customer': 0,
    'Attrited Customer': 1
}
df['Attrition_Flag'] = df['Attrition_Flag'].map(map_churn).astype('int8')
df = df.rename(columns={'Attrition_Flag': 'churn'})

X = df.drop(columns = ['churn'])
y = df['churn'].astype('int8').copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size = 0.2, 
    stratify = y,  
    random_state = 33
)

In [0]:
# Checking the proportions of the target variable
print(f'Shape of training: {X_train.shape}')
print(f'Shape of test: {X_test.shape}')

print('\n--- Churn Rate (Stratify Validation) ---')
print(f'Original: {y.mean():.2%}')
print(f'Train:    {y_train.mean():.2%}')
print(f'Test:    {y_test.mean():.2%}')

##### Construction of Data Preparation Pipelines
---


- For the data preparation stage, **two distinct pipelines** were developed: one designed for **linear models** and the other for **tree-based models**.


> This separation was adopted because each family of algorithms has its own preprocessing requirements, especially with regard to **scaling numerical variables** and **encoding categorical variables**.


- As discussed in the EDA, the initial decision was to remove only the variables **`equip`** and **`wireless`**, since they showed high correlation with **`equipmon`** and **`wiremon`**, respectively.


- The other highly correlated variables were deliberately retained at this initial stage.


> This decision allows the modeling process to empirically evaluate how different algorithms handle **multicollinearity**, **informational redundancy**, and the possible **marginal predictive gain** associated with these variables.


- In both pipelines, the data preparation flow followed the same general structure:


- **Variable type optimization**, with the objective of ensuring structural consistency, reducing memory usage, and adapting the data to the computational requirements of the algorithms.

- **Feature engineering**, with the creation of new derived variables based on findings from the exploratory analysis and domain knowledge.

- **Model-family-specific preprocessing**, respecting the technical particularities of each modeling approach and ensuring compatibility between the transformed data and the algorithms used.


---


##### Checking the numeric variables

In [0]:
pipeline_fg = Pipeline(
    steps = [
        ('types_vars', DtypeOptimizer()),
        ('fg', FeatureEngineer())
    ]
)
train_set_fg = pipeline_fg.fit_transform(X_train)
train_set_fg.head()

In [0]:
GraphicsData(train_set_fg.select_dtypes(include = ['float32'])).numerical_histograms()

#### Pre-processor for Linear Models

In [0]:
# Columns
one_hot_cols = [
    'gender', 'education_level', 'marital_status', 'income_category','card_category', 'trans_ct_bucket', 'trans_amt_bucket', 'tenure_bucket'
]

num_cols = [
    'utilization_amont', 'avg_utilization_ratio', 'total_revolving_bal', 'customer_age', 'dependent_count', 'months_on_book',
    'total_relationship_count', 'months_inactive_12_mon','contacts_count_12_mon', 'total_trans_ct'
]
 
skew_cols = [
    'credit_limit', 'total_amt_chng_q4_q1', 'total_trans_amt', 'total_ct_chng_q4_q1','avg_ticket', 'inactive_per_tenure','total_spending', 'change_gap_amt_ct', 
    'credit_revolving'
]

bin_cols = [
    'months_inactive_12_mon_0', 'contacts_count_12_mon_0','dependent_count_0', 'total_revolving_bal_0', 'total_amt_chng_q4_q1_0',
    'total_ct_chng_q4_q1_0', 'avg_utilization_ratio_0','low_activity_low_value_flag', 'is_pos_graduate', 'total_amt_q75','total_ct_q75'
]

drop_cols = [
    'clientnum', 
    'naive_bayes_classifier_attrition_flag_card_category_contacts_count_12_mon_dependent_count_education_level_months_inactive_12_mon_1',
    'naive_bayes_classifier_attrition_flag_card_category_contacts_count_12_mon_dependent_count_education_level_months_inactive_12_mon_2',
    'avg_open_to_buy'
]

# --------------------------------------------------------------------------------------------------------------------------------------#
# Pipelines

# Numerical columns with asymmetry
num_log = Pipeline(
    steps = [
        ('log', FunctionTransformer(np.log1p, feature_names_out = 'one-to-one')),
        ('std', StandardScaler()),
    ]
)

# One Hot Config
one_hot_encoder = OneHotEncoder(
    drop = 'first', 
    sparse_output = False, 
    handle_unknown = 'ignore', 
    feature_name_combiner = 'concat', 
    dtype = np.int8
)

linear_preprocess = ColumnTransformer(
    transformers = [
        ('num_cols', StandardScaler(),num_cols), 
        ('skew_cols', num_log, skew_cols),
        ('one_hot_cols', one_hot_encoder, one_hot_cols),
        ('bin_cols', 'passthrough', bin_cols),
        ('drop_cols', 'drop', drop_cols)
    ],
    remainder = 'passthrough',
    verbose_feature_names_out = False,
).set_output(transform = 'pandas')


linear_pipeline = Pipeline(
    steps = [
       ('types_vars', DtypeOptimizer()), 
       ('feature_eg', FeatureEngineer()),
       ('linear_preprocess', linear_preprocess)
    ]
)

X_train_prepared_linear = linear_pipeline.fit_transform(X_train)

In [0]:
X_train_prepared_linear.head()

#### Pre-processor for Tree Models

In [0]:

# Columns
one_hot_cols = [
    'gender', 'marital_status', 
]

ord_cols = [
    'education_level', 'income_category', 'card_category', 'trans_ct_bucket', 'trans_amt_bucket', 'tenure_bucket'
]


drop_cols = [
    'clientnum', 
    'naive_bayes_classifier_attrition_flag_card_category_contacts_count_12_mon_dependent_count_education_level_months_inactive_12_mon_1',
    'naive_bayes_classifier_attrition_flag_card_category_contacts_count_12_mon_dependent_count_education_level_months_inactive_12_mon_2',
    'avg_open_to_buy'
    ]


# One Hot Config
one_hot_encoder = OneHotEncoder(
    drop = 'first',
    sparse_output=False,
    handle_unknown = 'ignore',
    feature_name_combiner = 'concat',
    dtype = np.int8
)

# Create Ordinal pipeline

# Create Ordinal order
ordinal_features_order = {
'education_level': ['Uneducated', 'High School', 'College', 'Graduate', 'Post-Graduate', 'Doctorate', ],
'income_category': ['Less than $40K', '$40K - $60K', '$60K - $80K', '$80K - $120K', '$120K +', ],
'card_category': ['Blue', 'Silver', 'Gold', 'Platinum'],
'trans_ct_bucket': ['very_low_activity', 'low_activity', 'medium_activity', 'high_activity', 'very_high_activity'],
'trans_amt_bucket': ['very_low_activity', 'low_activity', 'medium_activity', 'high_activity', 'very_high_activity'], 
'tenure_bucket': ['new', 'developing', 'established', 'loyal']
} 

ordinal_categories = [
    ordinal_features_order['education_level'],
    ordinal_features_order['income_category'],
    ordinal_features_order['card_category'],
    ordinal_features_order['trans_ct_bucket'],
    ordinal_features_order['trans_amt_bucket'],
    ordinal_features_order['tenure_bucket'],
]

ordinal_pipeline = Pipeline(
    steps=[
        (
            'ordinal_encoder',
            OrdinalEncoder(
                categories = ordinal_categories,
                dtype = np.int8,
                handle_unknown='use_encoded_value', 
                unknown_value = -1
            ),
        ),
    ],
)


tree_preprocess = ColumnTransformer(
    transformers=[
        ('one_hot_cols', one_hot_encoder, one_hot_cols),
        ('ord_cols', ordinal_pipeline, ord_cols),
        ('drop_cols', 'drop', drop_cols),
    ],
    remainder = 'passthrough',
    verbose_feature_names_out = False,
).set_output(transform = 'pandas')
      
tree_pipeline = Pipeline(steps=[
    ('types_vars', DtypeOptimizer()),
    ('feature_eg', FeatureEngineer()),
    ('tree_preprocess', tree_preprocess),
])

X_train_prepared_tree = tree_pipeline.fit_transform(X_train)

In [0]:
X_train_prepared_tree.head()

### 4 - Modeling
---

##### Model Evaluation and Selection Strategy
---


- The primary metric defined for this project will be **AUC-ROC**.


> Since the problem involves **imbalanced classes** and the central objective is to develop a model capable of estimating the **probability of churn**, AUC-ROC is appropriate because it provides a comprehensive view of the model's discriminative ability between the two classes.


- As secondary metrics, **F1-score** and **recall for the churn class** will also be monitored.


> The **F1-score** will be used to evaluate the balance between **precision** and **recall**, while **recall for the churn class** will receive special attention, since correctly identifying customers at higher risk of churn is essential for guiding retention actions.


> This choice is strategically relevant, since **retaining a customer** through engagement and loyalty campaigns is approximately **five times less costly** than **acquiring a new customer**.


- Initial training will be conducted using **cross-validation**, with the objective of increasing the robustness of model evaluation at this stage of the project.


> **5 folds** will be used, and the selection of the best model will consider not only the **highest mean AUC-ROC**, but also the **lowest variability** among the results observed across the different folds, seeking consistent performance and greater generalization capability.


- Simpler models, such as **Logistic Regression** and **Decision Tree Classifier**, will be tested.


> This choice prioritizes solutions with **lower complexity** and **greater interpretability**, which becomes especially important given a relatively small dataset with only **200 records**, reducing the risk of **overfitting**.


- And more robust models, such as **Linear SVC** and **LightGBM**, will also be evaluated.


> The objective of this comparison is to verify whether the structure of the data benefits from algorithms with **greater modeling capacity**, making it possible to assess potential performance gains in relation to simpler approaches.


---


##### Cross-Validation

In [0]:
linear_models = {
    'Logistic Regression': LogisticRegression(),
    'Linear SVC': LinearSVC(),
    'KNeighbors Classifier': KNeighborsClassifier(),
}

eval_linear = classification_kfold_cv(
    models = linear_models,
    X_train = X_train_prepared_linear,
    y_train = y_train,
    n_folds = 5,
    scoring = 'roc_auc'
)
eval_linear

In [0]:
tree_models = {
    'Decision Tree ': DecisionTreeClassifier(),
    'Random Forest ': RandomForestClassifier(),
    'XGB ': XGBClassifier(),
    'LightGBM ': LGBMClassifier(verbosity = -1, ), # verbosity = -1 >> This is just a parameter that disables verbose e in LGBTM.
}
eval_tree = classification_kfold_cv(
    models = tree_models,
    X_train = X_train_prepared_tree,  
    y_train = y_train,
    n_folds = 5,
    
)
eval_tree

In [0]:
df_scores = pd.concat([eval_linear, eval_tree], ignore_index = True)
df_scores = df_scores[['avg_val_score', 'val_score_std', 'model']]


##### Models Scores 

In [0]:
GraphicsData(df_scores.sort_values('avg_val_score')).models_performance_barplots(models_col = 'model', figsize_per_row = 7,)

##### Initial results and direction for feature selection
---

- The results obtained so far can be considered **satisfactory**.

> The **LightGBM** model achieved an **AUC-ROC of 99.34%** and a **variance of 0.0017**, which is a relatively low value, indicating strong stability across the evaluated folds.

- The data in this dataset are the main drivers behind this excellent result, reflecting a high-quality and low-noise database.

> The selected features follow the principle of being extracted or defined before the churn event or before the customer’s continuation with the contracted services.

- Several variables demonstrated good separation ability between **churners** and **non-churners**.

> During the statistical tests performed in the exploratory analysis stage, it was observed that some of these variables, in addition to showing **statistical significance**, also had a relevant impact on the distinction between the groups.

- This behavior was also reflected in the initial training results, especially in the performance of **LightGBM**.

> As a **tree-based** model, its structure tends to handle nonlinear relationships, interactions between variables, and heterogeneous patterns in the data efficiently.

- In light of this scenario, **LightGBM** will be used in the **feature selection** stage.

> The objective will be to **simplify the model** and **remove redundant variables**, with the expectation of further improving predictive performance while also making the solution more **compact** and **interpretable**.
---


#### Feature Selection 
---


- **RFECV (Recursive Feature Elimination with Cross Validation)** will be used to select the most relevant variables for the model and remove redundant features or those with lower contribution.


> This approach allows the feature selection process to be conducted more robustly, combining recursive feature elimination with cross-validation to identify the configuration that best supports model performance.


- In this way, the goal is to obtain a **simpler**, more **compact**, and more **interpretable** solution, without significant loss in predictive performance.


> The expectation is to reduce the complexity of the feature space, preserve the variables with the strongest informational signal, and improve the efficiency of the modeling process while maintaining consistent performance across evaluations.


---


In [0]:
selector = RecursiveFeatureEliminator(
    estimator = LGBMClassifier(verbosity = -1, n_jobs = 1),
    scoring = 'roc_auc',
    n_folds = 5
)

selector.fit(X_train_prepared_tree, y_train)

X_train_selected = selector.transform(X_train_prepared_tree)

print(selector.selected_features_)
print(f'\n\nInitial Features: {len(X_train_prepared_tree.columns)}')
print(f'\nRemainder Features: {len(X_train_selected.columns)}')

In [0]:
X_train_selected.head()

##### Feature Selection Results with RFE
---


- The results obtained with the **Recursive Feature Eliminator** can be considered **satisfactory**.


> Of the initial **40 features**, **24 were retained** after the selection process, indicating a reduction in the feature space without losing the most relevant attributes for modeling.


- Dentre as variáveis selecionadas, a **`total_trans_ct`** e a **`total_trans_amt`**suas características derivadas se destacam.

> Esse resultado sugere que fatores associados à **a quantidade transações e o valor destas transações**  desempenham um papel importante na explicação da rotatividade de clientes na instituição.

---


#### Hyperparameter Optimization with Optuna
---


- In the **hyperparameter tuning** stage, **Optuna** will be used with the goal of making the hyperparameter optimization process more efficient than exhaustive methods such as **GridSearch**.


> This approach enables a more targeted exploration of the search space, prioritizing hyperparameter combinations with greater performance potential instead of evaluating all possibilities uniformly.


- Instead of testing combinations in a purely random way, **Optuna** uses sampling algorithms such as **TPE (Tree-structured Parzen Estimator)**.


> This mechanism takes into account the history of previous **trials** to identify more promising and less promising regions of the hyperparameter space, making the search process more adaptive and efficient.


- Considering the particular characteristics of this dataset, the definition of the search space was guided by a balance between hyperparameters that help mitigate limitations associated with the **small number of observations**.


> This strategy aims to control model complexity, reducing sensitivity to sample variation and minimizing the risk of excessive fitting to the training data.

---


In [0]:
stop

In [0]:


# scale_pos_weight
ratio = 8500 / 1627  # ---> 5.22
#'scale_pos_weight': ratio,
def objective(trial):
    params = {
        'is_unbalance': True,
        'boosting_type': 'gbdt',
        'subsample_freq': 1,
        'objective': 'binary',
        'metric': 'auc',
        'importance_type': 'gain',
        'n_estimators': 2000,
        'verbosity': -1,
        'random_state': 33,
        'n_jobs': 1,

        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 8, 128),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 60),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 10.0, log=True),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log = True),
        'subsample': trial.suggest_float('subsample', 0.5, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.95),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log = True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log = True),
        'path_smooth': trial.suggest_float('path_smooth', 0.0, 10.0),

    }

    model = LGBMClassifier(**params)

    cv = StratifiedKFold(
        n_splits = 5,
        shuffle = True,
        random_state = 33
    )

    scores = cross_val_score(
        model,
        X_train_selected,
        y_train,
        scoring = 'roc_auc',
        cv = cv,
        n_jobs = -1
    ) 

    trial.set_user_attr('auc_std', float(np.std(scores)))
    return float(np.mean(scores))


study = optuna.create_study(
    direction='maximize',
    study_name='lgbm_optuna_auc'
)

study.optimize(objective, n_trials = 100)

best_params = study.best_params
best_score = study.best_value
best_std = study.best_trial.user_attrs['auc_std']

print('\nBest Hyperparameters:', best_params)
print('Best Mean ROC-AUC:', best_score)
print('Best ROC-AUC Std:', best_std)

### 5 - Evaluation:

#### Final Training 

In [0]:
best_params = {
    'is_unbalance': True,
    'boosting_type': 'gbdt',
    'subsample_freq': 1,
    'objective': 'binary',
    'metric': 'auc',
    'importance_type': 'gain',
    'n_estimators': 2000,
    'verbosity': -1,
    'random_state': 33,
    'n_jobs': 1,
    
    'max_depth': 8, 
    'num_leaves': 12, 
    'min_child_samples': 13, 
    'min_child_weight': 0.02907698106052808, 
    'learning_rate': 0.014535185656296249, 
    'subsample': 0.6750883305967378, 
    'colsample_bytree': 0.6108215363210434, 
    'min_split_gain': 0.1480600865901492, 
    'reg_alpha': 0.012333922875086509, 
    'reg_lambda': 0.0023522045420049085, 
    'path_smooth': 0.6996556949819326
}

final_model = LGBMClassifier(** best_params)
final_model.fit(X_train_selected, y_train)

#### Testset Preparation

In [0]:
X_test_prepared_tree = tree_pipeline.transform(X_test)
X_test_selected =  selector.transform(X_test_prepared_tree)
X_test_selected.head()

#### Prediction

In [0]:
y_pred = final_model.predict(X_test_selected) 
y_prob = final_model.predict_proba(X_test_selected)[:, 1]

In [0]:
report_df, metrics_df = evaluation_scores(
    y_true=y_test,
    y_score=y_prob,
    threshold=0.5,
    title_prefix="Model - "
)


In [0]:
report_df

In [0]:
metrics_df

In [0]:
fig, axes, decile_df = plot_kde_predictions(
    y_true=y_test,
    y_score=y_prob
)




In [0]:
decile_df

In [0]:
from sklearn.metrics import roc_auc_score

auc_roc = roc_auc_score(y_test, y_prob)
print(f"AUC-ROC: {auc_roc:.4f}")

In [0]:
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = final_model.predict(X_test_selected)
y_prob = final_model.predict_proba(X_test_selected)[:, 1]

auc_roc = roc_auc_score(y_test, y_prob)
print(f"AUC-ROC: {auc_roc:.4f}")

cm = confusion_matrix(y_test, y_pred)
print("Matriz de Confusão:")
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Não Churn', 'Churn'])
disp.plot(cmap='Blues')
plt.title('Matriz de Confusão')
plt.show()


#### Key Observations:
---

#### 1. Technical Performance

  - **Explanatory Power (Real R² Score): 0.8844**

  - The model explains **88.4% of the variability** in CO2 emissions using the "Real World" scale (g/km). In the transformed mathematical space (Yeo-Johnson), the fit is even higher (**0.91**), confirming that the non-linear approach successfully captured the physical behavior of the data. This is a significant improvement over the simple univariate model (~0.80).

  - **Margin of Error (MAE): 13.03**

  - The **Mean Absolute Error** indicates that, on average, our predictions deviate by only **13.03 g/km** from the actual value. For a business context where emissions range up to 450 g/km, this represents a very high precision level (approx. 5% relative error), allowing for reliable carbon footprint estimation.

  - **Sensitivity to Large Errors (RMSE vs MAE):**

  * The **RMSE (20.87)** is controlled relative to the MAE (13.03). The gap of ~7.8 points is healthy. It indicates that while there are outliers (likely high-performance sports cars or heavy vehicles), the **Yeo-Johnson transformation** successfully mitigated the extreme penalties that usually distort linear models.
---

#### 2. Model Interpretation

  - The model utilizes a **Power Law** approach (Yeo-Johnson) rather than a simple straight line:

  - **Feature Dominance (Standardized Coefficients):**
  - **Fuel Consumption (Coefficient ~0.71):** This is the dominant driver. The stability analysis showed a variation of only **1.92%** (CV) for this weight, proving it is the most reliable predictor.
  - **Engine Size (Coefficient ~0.27):** This acts as a secondary adjustment factor. Even with a 0.82 correlation to fuel, the model successfully isolated its unique contribution with high stability (CV ~4.05%).

  - **The "Curved" Surface:**
  - Unlike a rigid linear equation, the model projects a **curved surface**. This means it understands that "efficiency" changes as engines get bigger. Physically, this represents the diminishing returns of combustion efficiency in larger engines, providing a much more realistic simulation than a simple linear slope.
---

#### 3. **Conclusion:**

- The Multiple Regression Model with Power Transformation represents the "State-of-the-Art" for this dataset.

- **Strengths:** - **Robustness:** The coefficient stability (CV < 5%) proves the model is immune to multicollinearity. 
- **Physical Coherence:** The residuals follow a near-perfect Gaussian distribution (Mean Bias ~0.81g), indicating that all deterministic signal has been captured.

- **Limitations:**

- **Interpretability:** Because the model operates in a transformed space, we cannot say "1 liter adds X grams" directly. We must use the inverse transformation to get real values.
- **Scope:** The slight residual spread at the high end (>350g/km) suggests that for extreme heavy-duty vehicles, a separate model or additional features (like vehicle weight) might be required.

### 6. Deployment:
---

In [0]:
# DEPLOY PACKAGE: We save everything in a dictionary to ensure integrity.
production_bundle = {
    'model': model, 
    'pt_X': X_scaler, 
    'pt_y': y_scaler,
}

# Saved in a single "pickle" file
joblib.dump(production_bundle, './artifacts/co2_pipeline_v2.pkl')

# Return
print('✅ Complete pipeline saved successfully!')

In [0]:
def predict_emission(engine_size, fuel_consumption):
    
    """
    Performs full inference with Yeo-Johnson pre- and post-processing.
    """
    # 1. Loading (Load the Complete Package)
    bundle = joblib.load('./artifacts/co2_pipeline_v2.pkl')
    model_loaded = bundle['model']
    pt_X_loaded = bundle['pt_X']
    pt_y_loaded = bundle['pt_y']

    # 2. Input Data Engineering
    input_data = pd.DataFrame(
        [[engine_size, fuel_consumption]],
        columns = ['ENGINESIZE', 'FUELCONSUMPTION_COMB']
    )

    # 3. Pre-processing
    input_transformed = pt_X_loaded.transform(input_data)

    # 4. Prediction
    prediction_transformed = model_loaded.predict(input_transformed)

    # 5. Post-processing (Reducing to Grams of CO2)
    prediction_real  = pt_y_loaded.inverse_transform(prediction_transformed.reshape(-1, 1))

    result_g_km = prediction_real[0][0]

    return result_g_km

# --- FINAL TEST (User Acceptance Test) ---

# Scenario: 2.0L car getting 8.5 L/100km
engine = 2.0
consumption = 8.5

try:
    prediction = predict_emission(engine, consumption)

    print(f'--- INFERENCE REPORT ---')
    print(f'Engine: {engine} L')
    print(f'Consumption: {consumption} L/100km')
    print(f'Predicted Emission: {prediction:.2f} g/km')

except Exception as e:
    print(f'Deployment error: {e}')